### **Instructions to run the apps**:

***Recommended way***: 
The easiest way to quickly and correctly run the apps in this assignment is to clone the github repo and follow below instructions:

1. Clone the github repo at https://github.com/Atul-Lohiya/assign_genai with command: git clone https://github.com/Atul-Lohiya/assign_genai.git
2. Switch to correct virtual environment. The apps were tested with python 3.12 and dependencies mentioned in requirements.txt.
3. Install all the dependencies present in requirements.txt with command: pip install -r requirements.txt
4. Run each streamlit app individually with streamlit run ./insurance_policy_agent/app.py and streamlit run ./Resume_Screening_RAG_Assistant/app.py respectively. For details of each app, go through the read.md files in respective folders

**Public streamlit apps**:
- Resume screening: https://assigngenai-resume-screening.streamlit.app/
- Insurance claim processing: https://assigngenai-insurance-claim-processing.streamlit.app/

***Alternate way***: Not recommended as it may cause errors due to missing/misplaced files/images.
1. Create folder structure as follows:
<pre>
  root
    |--insurance_policy_agent
    |--Resume_Screening_RAG_Assistant
</pre>

2. Add requirements.txt in root folder and copy contents from corresponding cell in this notebook

3. Create below files in insurance_policy_agent folder and copy corresponding code from below cells
    - app.py (streamlit app)
    - config.py (app config)
    - graph.py (lang graph)
    - llm.py (model creation)
    - nodes.py (nodes for langgraph)
    - state.py (graph state model)
    - README.md

4. Create below files in Resume_Screening_RAG_Assistant folder  and copy corresponding code from below cells
    - app.py (streamlit app)
    - config.toml (streamlit config for styling)
    - generate_sample_resumes.py (sample resume generator)
    - jd_data_scientist.txt (sample job description for data science role)
    - rag_engine.py (rag engine)
    - README.md

### Insurance Policy Agent Folder

In [ ]:
# File: app.py

'''

"""
streamlit_app.py
-----------------
Streamlit front-end for the Insurance Claim Processing Agent.

Run with:
    streamlit run streamlit_app.py
"""

import uuid
import random
import streamlit as st
from langgraph.types import Command
from graph import claim_graph
from config import REQUIRED_DOCUMENTS
import llm

print("Starting app")
print(st.session_state)

st.set_page_config(page_title="Insurance Claim Processing Agent", page_icon="📋", layout="wide")

# -----------------------------------------------------------------------
# Demo scenarios (matches the 5 example test cases from the brief)
# -----------------------------------------------------------------------
SCENARIOS = {
    "1. Complete claim, valid documents -> Auto Approve": dict(
        claim_id="CLM-1001",
        policy_number="POL-9001",
        claimant_name="Asha Rao",
        claim_type="Auto",
        claim_amount=2200.0,
        policy_coverage_limit=50000.0,
        incident_date="2026-05-10",
        policy_start_date="2024-01-01",
        policy_end_date="2027-01-01",
        prior_claims_last_year=0,
        submitted_documents=["claim_form", "police_report", "photos_of_damage", "repair_estimate"],
    ),
    "2. Missing required documents -> Reject": dict(
        claim_id="CLM-1002",
        policy_number="POL-9002",
        claimant_name="Daniel Kim",
        claim_type="Property",
        claim_amount=4500.0,
        policy_coverage_limit=80000.0,
        incident_date="2026-04-02",
        policy_start_date="2023-06-01",
        policy_end_date="2027-06-01",
        prior_claims_last_year=0,
        submitted_documents=["claim_form", "photos_of_damage"],  # missing estimate + ownership proof
    ),
    "3. Suspicious claim amount -> Human Review": dict(
        claim_id="CLM-1003",
        policy_number="POL-9003",
        claimant_name="Priya Menon",
        claim_type="Auto",
        claim_amount=46850.0,
        policy_coverage_limit=50000.0,  # ratio 0.94 -> fraud flag
        incident_date="2026-06-01",
        policy_start_date="2026-05-20",  # 12 days before incident -> recent policy flag
        policy_end_date="2027-05-20",
        prior_claims_last_year=1,
        submitted_documents=["claim_form", "police_report", "photos_of_damage", "repair_estimate"],
    ),
    "4. Expired policy -> Reject": dict(
        claim_id="CLM-1004",
        policy_number="POL-9004",
        claimant_name="Wei Zhang",
        claim_type="Health",
        claim_amount=1800.0,
        policy_coverage_limit=20000.0,
        incident_date="2026-07-01",
        policy_start_date="2023-01-01",
        policy_end_date="2025-12-31",  # policy ended before incident
        prior_claims_last_year=1,
        submitted_documents=["claim_form", "medical_bill", "doctor_report"],
    ),
    "5. High-value claim, valid documents -> Human Review": dict(
        claim_id="CLM-1005",
        policy_number="POL-9005",
        claimant_name="Grace Odhiambo",
        claim_type="Property",
        claim_amount=32000.0,  # above HIGH_VALUE_CLAIM_THRESHOLD
        policy_coverage_limit=100000.0,
        incident_date="2026-03-15",
        policy_start_date="2022-01-01",
        policy_end_date="2027-01-01",
        prior_claims_last_year=0,
        submitted_documents=["claim_form", "photos_of_damage", "repair_estimate", "ownership_proof"],
    ),
}

def sync_claim_to_widgets(claim_data):
    st.session_state["claim_id"] = claim_data["claim_id"]
    st.session_state["policy_number"] = claim_data["policy_number"]
    st.session_state["claimant_name"] = claim_data["claimant_name"]
    st.session_state["claim_type"] = claim_data["claim_type"]
    st.session_state["claim_amount"] = claim_data["claim_amount"]
    st.session_state["policy_coverage_limit"] = claim_data["policy_coverage_limit"]
    st.session_state["incident_date"] = claim_data["incident_date"]
    st.session_state["policy_start_date"] = claim_data["policy_start_date"]
    st.session_state["policy_end_date"] = claim_data["policy_end_date"]
    st.session_state["prior_claims_last_year"] = claim_data["prior_claims_last_year"]
    st.session_state["submitted_documents"] = claim_data["submitted_documents"]

# -----------------------------------------------------------------------
# Run the graph
# -----------------------------------------------------------------------
def process_claim(claim_data: dict):
    thread_id = str(uuid.uuid4())
    st.session_state.thread_id = thread_id
    config = {"configurable": {"thread_id": thread_id}}

    try:
        result = claim_graph.invoke(claim_data, config=config)
    except RuntimeError as e:
        st.session_state.result_state = None
        st.session_state.llm_error = str(e)
        return

    st.session_state.llm_error = None
    st.session_state.result_state = result

    if "__interrupt__" in result:
        st.session_state.awaiting_human = True
        st.session_state.interrupt_payload = result["__interrupt__"][0].value
    else:
        st.session_state.awaiting_human = False
        st.session_state.interrupt_payload = None


def resume_with_human_decision(decision: str, notes: str):
    config = {"configurable": {"thread_id": st.session_state.thread_id}}
    try:
        result = claim_graph.invoke(
            Command(resume={"decision": decision, "notes": notes}), config=config
        )
    except RuntimeError as e:
        st.session_state.llm_error = str(e)
        return
    st.session_state.llm_error = None
    st.session_state.result_state = result
    st.session_state.awaiting_human = False
    st.session_state.interrupt_payload = None

def submit_claim():
    print("submit click handler executed")
    st.session_state.claim_details_expander = False

    claim_data = {
                "claim_id": st.session_state.claim_id,
                "policy_number": st.session_state.policy_number,
                "claimant_name": st.session_state.claimant_name,
                "claim_type": st.session_state.claim_type,
                "claim_amount": st.session_state.claim_amount,
                "policy_coverage_limit": st.session_state.policy_coverage_limit,
                "incident_date": st.session_state.incident_date,
                "policy_start_date": st.session_state.policy_start_date,
                "policy_end_date": st.session_state.policy_end_date,
                "prior_claims_last_year": st.session_state.prior_claims_last_year,
                "submitted_documents": st.session_state.submitted_documents,
            }
    
    print("Processing claim")
    process_claim(claim_data)
    print("Claim processing completed")

    # Store submitted claim
    st.session_state.claim_data = claim_data

    # Optional: store a flag that results are available
    st.session_state.claim_submitted = True

def reset_assessment():
    print("Resetting assessment")
    st.session_state.claim_submitted = False
    st.session_state.claim_data = {}


CLAIM_TYPES = list(REQUIRED_DOCUMENTS.keys())

# -----------------------------------------------------------------------
# Session state
# -----------------------------------------------------------------------
if "thread_id" not in st.session_state:
    st.session_state.thread_id = None
if "result_state" not in st.session_state:
    st.session_state.result_state = None
if "awaiting_human" not in st.session_state:
    st.session_state.awaiting_human = False
if "interrupt_payload" not in st.session_state:
    st.session_state.interrupt_payload = None


st.title("Insurance Claim Processing Agent")
st.caption("Built with LangGraph - parallel verification/eligibility/fraud checks, conditional routing, and human-in-the-loop escalation.")

scenario_name = None
claim_data = None

# -----------------------------------------------------------------------
# Sidebar: LLM provider + API key (never written to disk -- kept only in
# this browser session's memory and passed straight to the API call).
# -----------------------------------------------------------------------
with st.sidebar:
    st.header("LLM Settings")
    provider_label = st.selectbox(
        "Provider",
        ["Mock (rule based, no key needed)", "DeepSeek", "OpenAI"],
        help="Pick which model powers the 4 reasoning agents (Document Verification, "
        "Eligibility, Fraud Detection, Claim Summary). The Decision router itself is "
        "rule-based and doesn't call the LLM.",
    )

    api_key = None
    model_override = None
    provider_key = "mock"

    if provider_label == "DeepSeek":
        provider_key = "deepseek"
        api_key = st.text_input(
            "DeepSeek API key",
            type="password",
            placeholder="sk-...",
            help="Get one at platform.deepseek.com. Stored only in memory for this session.",
        )
        model_override = st.text_input("Model", value="deepseek-chat")
    elif provider_label == "OpenAI":
        provider_key = "openai"
        api_key = st.text_input(
            "OpenAI API key",
            type="password",
            placeholder="sk-...",
            help="Stored only in memory for this session.",
        )
        model_override = st.text_input("Model", value="gpt-4o-mini")

    llm.configure(provider_key, api_key or None, model_override or None)

    if llm.is_live():
        st.success(f"Connected - using **{llm.current_provider()}** for live reasoning.", icon="✅")
    else:
        st.info(
            "Running in **mock mode** - no key entered, so the 4 reasoning agents use "
            "built-in rule-based logic instead of a live LLM call. Change the mode and enter a key above to "
            "switch to real reasoning.",
            icon="ℹ️",
        )

    st.divider()

    st.header("Claim Scenario")
    mode = st.radio("Select a mode", ["Preloaded demo scenario", "Custom claim"], key="claim_mode", on_change=reset_assessment)

    if mode == "Preloaded demo scenario":
        scenario_name = st.selectbox("Scenario", list(SCENARIOS.keys()), on_change=reset_assessment)


# ---------------------------------------------------------
# Initialize / reset sample claim data for selected scenario
# ---------------------------------------------------------

if mode == "Preloaded demo scenario":
    claim_data = dict(SCENARIOS[scenario_name])
else:
    # Auto-generate identifiers for custom claim
    claim_data = dict(
        claim_id=f"CLM-{random.randint(0, 9999):04d}",
        policy_number=f"POL-{random.randint(0, 9999):04d}",
        claimant_name="",
        claim_type="Auto",
        claim_amount=0.0,
        policy_coverage_limit=0.0,
        incident_date="",
        policy_start_date="",
        policy_end_date="",
        prior_claims_last_year=0,
        submitted_documents=[],
    )

if (
    "claim_scenario" not in st.session_state
    or st.session_state.claim_scenario != scenario_name
):
    st.session_state.claim_data = claim_data
    st.session_state.claim_scenario = scenario_name
    sync_claim_to_widgets(claim_data)


# ---------------------------------------------------------
# Claim Input Section
# ---------------------------------------------------------

with st.expander(
    "Claim Details",
    expanded=not st.session_state.get("claim_submitted", False),
    key="claim_details_expander",
    on_change = "rerun"
    
):

    # Row 1
    col1, col2, col3, col4 = st.columns(4)

    with col1:
        claim_id = st.text_input(
            "Claim ID",
            key="claim_id"
        )

    with col2:
        policy_number = st.text_input(
            "Policy Number",
            key="policy_number"
        )

    with col3:
        claimant_name = st.text_input(
            "Claimant Name",
            key="claimant_name"
        )

    with col4:
        claim_type = st.selectbox(
            "Claim Type",
            CLAIM_TYPES,
            key="claim_type"
        )


    # Row 2
    col1, col2, col3, col4 = st.columns(4)

    with col1:
        claim_amount = st.number_input(
            "Claim Amount ($)",
            min_value=0.0,
            step=100.0,
            key="claim_amount"
        )

    with col2:
        policy_coverage_limit = st.number_input(
            "Policy Coverage Limit ($)",
            min_value=0.0,
            step=1000.0,
            key="policy_coverage_limit"
        )

    with col3:
        incident_date = st.text_input(
            "Incident Date (YYYY-MM-DD)",
            key="incident_date"
        )

    with col4:
        policy_start_date = st.text_input(
            "Policy Start Date (YYYY-MM-DD)",
            key="policy_start_date"
        )


    # Row 3
    col1, col2, col3, col4 = st.columns(4)

    with col1:
        policy_end_date = st.text_input(
            "Policy End Date (YYYY-MM-DD)",
            key="policy_end_date"
        )

    with col2:
        prior_claims_last_year = st.number_input(
            "Prior Claims Last Year",
            min_value=0,
            step=1,
            key="prior_claims_last_year"
        )

    # Determine documents based on selected claim type
    available_docs = REQUIRED_DOCUMENTS.get(claim_type, [])

    with col3:
        submitted_documents = st.multiselect(
            "Submitted Documents",
            options=available_docs,
            key="submitted_documents"
        )

    # Fourth column intentionally left empty
    with col4:
        st.empty()


# -----------------------------------------------------
# Submit button
# -----------------------------------------------------
submit_clicked = st.button(
    "Submit Claim",
    type="primary",
    use_container_width=True,
    on_click=submit_claim
)


# ---------------------------------------------------------
# Process submission
# ---------------------------------------------------------

if "llm_error" not in st.session_state:
    st.session_state.llm_error = None

if st.session_state.llm_error:
    st.error(f"{st.session_state.llm_error}")
    
# ---------------------------------------------------------
# Results Section - Always visible
# ---------------------------------------------------------

with st.container(border=True):

    st.subheader("Claim Assessment")

    if st.session_state.get("claim_submitted", False):

        claim_data = st.session_state.claim_data

        # -----------------------------------------------------------------------
        # Display results
        # -----------------------------------------------------------------------
        result = st.session_state.result_state

        if result:
            st.divider()
            col1, col2, col3 = st.columns(3)
            with col1:
                st.subheader("Document Verification")
                st.write("Verified" if result.get("documents_verified") else "Not Verified")
                if result.get("missing_documents"):
                    st.write(f"Missing: {', '.join(result['missing_documents'])}")
                st.caption(result.get("document_notes", ""))

            with col2:
                st.subheader("Eligibility Check")
                st.write("Eligible" if result.get("eligibility_status") else "Not Eligible")
                st.caption(result.get("eligibility_notes", ""))

            with col3:
                st.subheader("Fraud Detection")
                score = result.get("fraud_risk_score", 0)
                st.metric("Fraud Risk Score", f"{score}/100")
                for flag in result.get("fraud_flags", []):
                    st.caption(f"{flag}")

            st.divider()
            st.subheader("Claim Summary")
            st.write(result.get("claim_summary", ""))

            st.divider()

            if st.session_state.awaiting_human and st.session_state.interrupt_payload:
                payload = st.session_state.interrupt_payload
                st.warning(
                    f"**Human review required** - {payload.get('message', '')}\n\n"
                    f"Reason for escalation: {result.get('decision_reason', '')}"
                )
                with st.form("human_review_form"):
                    notes = st.text_area("Reviewer notes")
                    c1, c2 = st.columns(2)
                    approve = c1.form_submit_button("Approve Claim", use_container_width=True)
                    reject = c2.form_submit_button("Reject Claim", use_container_width=True)

                if approve:
                    resume_with_human_decision("approved", notes)
                    st.rerun()
                if reject:
                    resume_with_human_decision("rejected", notes)
                    st.rerun()

            else:
                final_status = result.get("final_status", "Pending")
                decision_reason = result.get("decision_reason", "")
                if final_status == "Approved":
                    st.success(f"### Final Status: {final_status}")
                elif final_status == "Rejected":
                    st.error(f"### Final Status: {final_status}")
                else:
                    st.info(f"### Final Status: {final_status}")
                st.caption(f"Decision reasoning: {decision_reason}")
                if result.get("human_decision"):
                    st.caption(
                        f"Human reviewer decision: **{result['human_decision']}** - notes: "
                        f"{result.get('human_notes') or '(none)'}"
                    )

            with st.expander("Full agent execution trace"):
                for line in result.get("trace", []):
                    st.text(line)

            with st.expander("Raw final state (debug)"):
                st.json({k: v for k, v in result.items() if k != "__interrupt__"})

        else:
            st.info("Choose a scenario or build a custom claim in the sidebar, then click **Run Claim Through Agent**.")

    else:
        st.info("Submit a claim to view the assessment.")

print("Ending app")
st.session_state.rerun = False


'''

In [ ]:
# File: config.py

'''

"""
config.py
---------
Static business rules that a real insurer would keep in a database.
Kept simple/hard-coded here for demo purposes.
"""

REQUIRED_DOCUMENTS = {
    "Auto": ["claim_form", "police_report", "photos_of_damage", "repair_estimate"],
    "Health": ["claim_form", "medical_bill", "doctor_report"],
    "Property": ["claim_form", "photos_of_damage", "repair_estimate", "ownership_proof"],
    "Travel": ["claim_form", "boarding_pass", "receipt"],
}

# Claim types each policy plan covers (demo simplification: one plan "Standard")
COVERED_CLAIM_TYPES = {"Auto", "Health", "Property", "Travel"}

# Thresholds
FRAUD_ESCALATION_THRESHOLD = 60      # >= this score -> escalate for human review
FRAUD_AUTO_REJECT_THRESHOLD = 90     # >= this score -> auto reject (extremely high confidence fraud)
HIGH_VALUE_CLAIM_THRESHOLD = 15000   # claims at/above this amount always go to human review
RECENT_POLICY_WINDOW_DAYS = 14       # incident within N days of policy start looks suspicious


'''

In [ ]:
# File: graph.py

'''

"""
graph.py
--------
Wires the five agents into a LangGraph StateGraph:

                        ┌──────────────────────────┐
                        │           START          │
                        └─────────────┬────────────┘
              ┌────────────────────────┼─────────────────────────┐
              ▼                        ▼                         ▼
   Document Verification      Eligibility Check          Fraud Detection      <- run in PARALLEL
              └────────────────────────┼─────────────────────────┘
                                       ▼
                                    merge
                                       ▼
                              Claim Summary Agent
                                       ▼
                               Decision Router
                     ┌────────────────┼────────────────┐
                     ▼                ▼                ▼
              auto_approve        auto_reject     Human Approval Agent
                     │                │           (interrupt / HITL)
                     ▼                ▼                 ▼
                    END              END               END

A checkpointer (MemorySaver) is required for the interrupt()-based
human-in-the-loop step to work -- it lets the graph pause after
`human_approval_node` calls interrupt() and resume later from the same
point once the human's decision is supplied via Command(resume=...).
"""

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

from state import ClaimState
from nodes import (
    document_verification_node,
    eligibility_check_node,
    fraud_detection_node,
    merge_node,
    claim_summary_node,
    decision_node,
    route_after_decision,
    human_approval_node,
    auto_approve_node,
    auto_reject_node,
)


def build_graph():
    graph = StateGraph(ClaimState)

    # Nodes
    graph.add_node("document_verification", document_verification_node)
    graph.add_node("eligibility_check", eligibility_check_node)
    graph.add_node("fraud_detection", fraud_detection_node)
    graph.add_node("merge", merge_node)
    graph.add_node("claim_summary", claim_summary_node)
    graph.add_node("decision", decision_node)
    graph.add_node("auto_approve", auto_approve_node)
    graph.add_node("auto_reject", auto_reject_node)
    graph.add_node("human_approval", human_approval_node)

    # --- Parallel fan-out from START ---------------------------------
    graph.add_edge(START, "document_verification")
    graph.add_edge(START, "eligibility_check")
    graph.add_edge(START, "fraud_detection")

    # --- Fan-in: all three parallel branches must complete before merge
    graph.add_edge("document_verification", "merge")
    graph.add_edge("eligibility_check", "merge")
    graph.add_edge("fraud_detection", "merge")

    # --- Sequential from here -----------------------------------------
    graph.add_edge("merge", "claim_summary")
    graph.add_edge("claim_summary", "decision")

    # --- Conditional routing based on decision -------------------------
    graph.add_conditional_edges(
        "decision",
        route_after_decision,
        {
            "auto_approve": "auto_approve",
            "reject": "auto_reject",
            "human_review": "human_approval",
        },
    )

    graph.add_edge("auto_approve", END)
    graph.add_edge("auto_reject", END)
    graph.add_edge("human_approval", END)

    checkpointer = MemorySaver()
    compiled = graph.compile(checkpointer=checkpointer)
    return compiled


# A module-level singleton so Streamlit (which reruns the script on every
# interaction) doesn't rebuild the graph unnecessarily.
claim_graph = build_graph()


'''

In [ ]:
# File: llm.py

'''

"""
llm.py
------
Thin LLM wrapper used by every agent node.

Supports three providers, chosen at runtime (from the Streamlit sidebar,
or via environment variables for non-UI use):

  - "mock"      : no API key needed. Deterministic rule-flavoured
                   reasoning so the whole graph runs end-to-end for free.
  - "openai"    : OpenAI models (e.g. gpt-4o-mini), via langchain_openai.
  - "deepseek"  : DeepSeek models (e.g. deepseek-chat). DeepSeek exposes
                   an OpenAI-compatible /chat/completions endpoint, so we
                   reuse langchain_openai.ChatOpenAI and just point it at
                   DeepSeek's base_url.

The key is NEVER written to disk or baked into the code -- it's held
only in memory for the current process/session (see `configure()` and
how streamlit_app.py calls it from a password-masked sidebar field).

Every node calls `call_llm(system_prompt, user_prompt)` and gets back a
string, regardless of which provider is active, so nothing in nodes.py
needs to change when you switch providers.
"""

import os
import json

DEEPSEEK_BASE_URL = "https://api.deepseek.com"

DEFAULT_MODELS = {
    "openai": "gpt-4o-mini",
    "deepseek": "deepseek-chat",
}

# Current runtime configuration (mutated by configure(), read by call_llm()).
_state = {
    "provider": "mock",
    "api_key": None,
    "model": None,
    "client": None,
}


def configure(provider: str, api_key: str | None = None, model: str | None = None):
    """
    Set which LLM provider/key/model subsequent call_llm() calls should use.
    Called once per Streamlit run (or once at startup for non-UI use).

    provider: "mock" | "openai" | "deepseek"
    api_key:  the user's own API key (required for "openai"/"deepseek")
    model:    optional override, otherwise a sensible default per provider
    """
    provider = (provider or "mock").lower()
    _state["provider"] = provider
    _state["api_key"] = api_key
    _state["client"] = None  # rebuilt lazily below

    if provider == "mock" or not api_key:
        _state["provider"] = "mock"
        return

    _state["model"] = model or DEFAULT_MODELS.get(provider, "gpt-4o-mini")

    try:
        from langchain_openai import ChatOpenAI

        kwargs = dict(model=_state["model"], temperature=0, api_key=api_key)
        if provider == "deepseek":
            kwargs["base_url"] = DEEPSEEK_BASE_URL

        _state["client"] = ChatOpenAI(**kwargs)
    except Exception as e:  # pragma: no cover
        print(f"[llm.py] Could not initialize {provider} client, falling back to mock: {e}")
        _state["provider"] = "mock"
        _state["client"] = None


def current_provider() -> str:
    return _state["provider"]


def is_live() -> bool:
    return _state["provider"] != "mock" and _state["client"] is not None


# ---------------------------------------------------------------------
# Bootstrap from environment variables so the graph also works outside
# Streamlit (e.g. scripts, tests) without any explicit configure() call.
# ---------------------------------------------------------------------
if os.environ.get("DEEPSEEK_API_KEY"):
    configure("deepseek", os.environ["DEEPSEEK_API_KEY"], os.environ.get("CLAIM_AGENT_MODEL"))
elif os.environ.get("OPENAI_API_KEY"):
    configure("openai", os.environ["OPENAI_API_KEY"], os.environ.get("CLAIM_AGENT_MODEL"))


def call_llm(system_prompt: str, user_prompt: str) -> str:
    """Call the configured LLM and return raw text content."""
    if is_live():
        from langchain_core.messages import SystemMessage, HumanMessage

        try:
            resp = _state["client"].invoke(
                [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
            )
            return resp.content
        except Exception as e:
            # Surface the real error (bad key, quota, network) rather than
            # silently mock-answering, so the user knows to check their key.
            raise RuntimeError(
                f"{_state['provider']} API call failed: {e}. "
                "Check your API key/quota, or switch to Mock mode in the sidebar."
            ) from e

    # ---- Mock fallback -----------------------------------------------
    # Deterministic, rule-flavoured "reasoning" so the demo works without
    # any API key. Each node passes enough context in user_prompt that we
    # can produce a plausible structured answer.
    return _mock_reasoning(system_prompt, user_prompt)


def _mock_reasoning(system_prompt: str, user_prompt: str) -> str:
    """
    Very small heuristic engine that mimics what an LLM would return for
    each of our node types, based on keywords baked into the prompts by
    the calling node. Returns JSON text in all cases, matching what the
    real LLM is instructed to return.
    """
    if "DOCUMENT_VERIFICATION_TASK" in system_prompt:
        missing = "MISSING_DOCS::" in user_prompt and "MISSING_DOCS::none" not in user_prompt
        if missing:
            return json.dumps(
                {
                    "verified": False,
                    "notes": "One or more required documents were not found in the submission. "
                    "The claim cannot proceed to eligibility review until they are provided.",
                }
            )
        return json.dumps(
            {
                "verified": True,
                "notes": "All required documents for this claim type are present and appear "
                "consistent with the claim details provided.",
            }
        )

    if "ELIGIBILITY_TASK" in system_prompt:
        if "POLICY_EXPIRED::true" in user_prompt:
            return json.dumps(
                {
                    "eligible": False,
                    "notes": "The policy was not active on the incident date -- the policy period "
                    "had already ended before the loss occurred.",
                }
            )
        if "COVERAGE_MISMATCH::true" in user_prompt:
            return json.dumps(
                {
                    "eligible": False,
                    "notes": "The claim type is not covered under the policy's current plan.",
                }
            )
        return json.dumps(
            {
                "eligible": True,
                "notes": "The policy was active on the date of the incident and the claim type "
                "falls within the covered categories.",
            }
        )

    if "FRAUD_TASK" in system_prompt:
        score = 10
        flags = []
        if "AMOUNT_RATIO_HIGH::true" in user_prompt:
            score += 45
            flags.append("Claim amount is unusually high relative to policy coverage limits")
        if "RECENT_POLICY_START::true" in user_prompt:
            score += 30
            flags.append("Incident occurred shortly after the policy was purchased")
        if "ROUND_NUMBER_AMOUNT::true" in user_prompt:
            score += 10
            flags.append("Claim amount is a suspiciously round figure")
        if "PRIOR_CLAIMS_HIGH::true" in user_prompt:
            score += 15
            flags.append("Claimant has an elevated recent claims history")
        score = min(score, 97)
        if not flags:
            flags.append("No significant fraud indicators detected")
        return json.dumps({"fraud_risk_score": score, "flags": flags})

    if "SUMMARY_TASK" in system_prompt:
        return json.dumps(
            {
                "summary": "Mock summary generated without a live LLM connection. In production "
                "this would be a concise, underwriter-ready narrative combining document "
                "status, eligibility findings, and fraud risk into a recommendation."
            }
        )

    return json.dumps({"note": "mock response"})


'''

In [ ]:
# File: nodes.py

'''

"""
nodes.py
--------
The five required agents, implemented as LangGraph nodes:

  1. document_verification_node
  2. eligibility_check_node
  3. fraud_detection_node
  4. claim_summary_node
  5. human_approval_node

Plus two small support nodes:
  - merge_node        (fan-in point after the 3 parallel checks)
  - decision_node      (rule-based router that reads the 3 parallel results)

Nodes 1-3 are fanned out in parallel from the graph's entry point (see
graph.py) and fan back in to `merge_node` before summary/decision.
"""

import json
from datetime import datetime

from langgraph.types import interrupt

from state import ClaimState
from config import (
    REQUIRED_DOCUMENTS,
    COVERED_CLAIM_TYPES,
    FRAUD_ESCALATION_THRESHOLD,
    FRAUD_AUTO_REJECT_THRESHOLD,
    HIGH_VALUE_CLAIM_THRESHOLD,
    RECENT_POLICY_WINDOW_DAYS,
)
from llm import call_llm


def _parse_date(s: str):
    return datetime.strptime(s, "%Y-%m-%d")


# ---------------------------------------------------------------------
# 1. Document Verification Agent
# ---------------------------------------------------------------------
def document_verification_node(state: ClaimState) -> dict:
    claim_type = state["claim_type"]
    required = REQUIRED_DOCUMENTS.get(claim_type, ["claim_form"])
    submitted = set(state.get("submitted_documents", []))
    missing = [d for d in required if d not in submitted]

    system_prompt = (
        "You are the Document Verification Agent for an insurance claims system. "
        "DOCUMENT_VERIFICATION_TASK. Decide if all required documents are present, "
        "given the required list and the submitted list. Respond ONLY as JSON with "
        "keys 'verified' (bool) and 'notes' (string)."
    )
    user_prompt = (
        f"Claim type: {claim_type}\n"
        f"Required documents: {required}\n"
        f"Submitted documents: {sorted(submitted)}\n"
        f"MISSING_DOCS::{'none' if not missing else ','.join(missing)}"
    )

    raw_llm_response = call_llm(system_prompt, user_prompt)
    try:
        parsed = json.loads(raw_llm_response)
    except Exception:
        parsed = {"verified": not missing, "notes": raw_llm_response}

    verified = bool(parsed.get("verified", not missing)) and not missing
    notes = parsed.get("notes", "")

    return {
        "required_documents": required,
        "missing_documents": missing,
        "documents_verified": verified,
        "document_notes": notes,
        "trace": [
            f"[Document Verification Agent] verified={verified}, missing={missing or 'none'}"
        ],
    }


# ---------------------------------------------------------------------
# 2. Eligibility Check Agent
# ---------------------------------------------------------------------
def eligibility_check_node(state: ClaimState) -> dict:
    incident_date = _parse_date(state["incident_date"])
    policy_start_date = _parse_date(state["policy_start_date"])
    policy_end_date = _parse_date(state["policy_end_date"])

    is_policy_expired = not (policy_start_date <= incident_date <= policy_end_date)
    coverage_mismatch = state["claim_type"] not in COVERED_CLAIM_TYPES

    system_prompt = (
        "You are the Eligibility Check Agent for an insurance claims system. "
        "ELIGIBILITY_TASK. Decide if the claim is eligible for coverage based on "
        "policy dates and claim type. Respond ONLY as JSON with keys 'eligible' (bool) "
        "and 'notes' (string)."
    )
    user_prompt = (
        f"Claim type: {state['claim_type']}\n"
        f"Incident date: {state['incident_date']}\n"
        f"Policy period: {state['policy_start_date']} to {state['policy_end_date']}\n"
        f"POLICY_EXPIRED::{'true' if is_policy_expired else 'false'}\n"
        f"COVERAGE_MISMATCH::{'true' if coverage_mismatch else 'false'}"
    )

    raw_llm_response = call_llm(system_prompt, user_prompt)
    try:
        parsed = json.loads(raw_llm_response)
    except Exception:
        parsed = {"eligible": not (is_policy_expired or coverage_mismatch), "notes": raw_llm_response}

    eligible = bool(parsed.get("eligible", True)) and not is_policy_expired and not coverage_mismatch
    notes = parsed.get("notes", "")

    return {
        "eligibility_status": eligible,
        "eligibility_notes": notes,
        "trace": [f"[Eligibility Check Agent] eligible={eligible}"],
    }


# ---------------------------------------------------------------------
# 3. Fraud Detection Agent
# ---------------------------------------------------------------------
def fraud_detection_node(state: ClaimState) -> dict:
    claim_amount = state["claim_amount"]
    coverage_limit = state.get("policy_coverage_limit", claim_amount) or claim_amount
    ratio_high = coverage_limit > 0 and (claim_amount / coverage_limit) >= 0.9
    policy_start_date = _parse_date(state["policy_start_date"])
    incident_date = _parse_date(state["incident_date"])
    recent_policy = 0 <= (incident_date - policy_start_date).days <= RECENT_POLICY_WINDOW_DAYS
    round_number = claim_amount % 1000 == 0 and claim_amount > 0
    prior_claims_high = state.get("prior_claims_last_year", 0) >= 3

    system_prompt = (
        "You are the Fraud Detection Agent for an insurance claims system. "
        "FRAUD_TASK. Estimate a fraud_risk_score from 0-100 and list any flags. "
        "Respond ONLY as JSON with keys 'fraud_risk_score' (int) and 'flags' (list of strings)."
    )
    user_prompt = (
        f"Claim amount: {claim_amount}, Policy coverage limit: {coverage_limit}\n"
        f"AMOUNT_RATIO_HIGH::{'true' if ratio_high else 'false'}\n"
        f"RECENT_POLICY_START::{'true' if recent_policy else 'false'}\n"
        f"ROUND_NUMBER_AMOUNT::{'true' if round_number else 'false'}\n"
        f"PRIOR_CLAIMS_HIGH::{'true' if prior_claims_high else 'false'}"
    )

    raw_llm_response = call_llm(system_prompt, user_prompt)
    try:
        parsed = json.loads(raw_llm_response)
    except Exception:
        parsed = {"fraud_risk_score": 50 if ratio_high or recent_policy else 10, "flags": [raw_llm_response]}

    score = int(parsed.get("fraud_risk_score", 10))
    flags = parsed.get("flags", [])

    return {
        "fraud_risk_score": score,
        "fraud_flags": flags,
        "fraud_notes": "; ".join(flags),
        "trace": [f"[Fraud Detection Agent] risk_score={score}, flags={flags}"],
    }


# ---------------------------------------------------------------------
# Fan-in merge node (no-op placeholder so we have one clean join point)
# ---------------------------------------------------------------------
def merge_node(state: ClaimState) -> dict:
    return {"trace": ["[Merge] all parallel checks complete -- proceeding to summary"]}


# ---------------------------------------------------------------------
# 4. Claim Summary Agent
# ---------------------------------------------------------------------
def claim_summary_node(state: ClaimState) -> dict:
    system_prompt = (
        "You are the Claim Summary Agent for an insurance claims system. "
        "SUMMARY_TASK. Write a concise, professional 3-5 sentence summary of the claim "
        "for an underwriter, combining document status, eligibility, and fraud risk. "
        "Respond ONLY as JSON with key 'summary' (string)."
    )
    user_prompt = (
        f"Claim ID: {state['claim_id']}\n"
        f"Claimant: {state['claimant_name']}\n"
        f"Claim type: {state['claim_type']}, Amount: {state['claim_amount']}\n"
        f"Documents verified: {state['documents_verified']} "
        f"(missing: {state.get('missing_documents') or 'none'})\n"
        f"Document notes: {state.get('document_notes', '')}\n"
        f"Eligibility: {state['eligibility_status']} -- {state.get('eligibility_notes', '')}\n"
        f"Fraud risk score: {state['fraud_risk_score']} -- flags: {state.get('fraud_flags', [])}"
    )

    raw_llm_response = call_llm(system_prompt, user_prompt)
    try:
        parsed = json.loads(raw_llm_response)
        summary = parsed.get("summary", raw_llm_response)
    except Exception:
        summary = raw_llm_response

    return {
        "claim_summary": summary,
        "trace": ["[Claim Summary Agent] summary generated"],
    }


# ---------------------------------------------------------------------
# Decision node (rule-based router -- reads outputs of the 3 parallel agents)
# ---------------------------------------------------------------------
def decision_node(state: ClaimState) -> dict:
    if not state["documents_verified"]:
        decision = "reject"
        reason = f"Missing required documents: {', '.join(state.get('missing_documents', []))}."
    elif not state["eligibility_status"]:
        decision = "reject"
        reason = state.get("eligibility_notes", "Policy is not eligible for this claim.")
    elif state["fraud_risk_score"] >= FRAUD_AUTO_REJECT_THRESHOLD:
        decision = "reject"
        reason = f"Fraud risk score of {state['fraud_risk_score']} exceeds the auto-reject threshold."
    elif (
        state["fraud_risk_score"] >= FRAUD_ESCALATION_THRESHOLD
        or state["claim_amount"] >= HIGH_VALUE_CLAIM_THRESHOLD
    ):
        decision = "human_review"
        reasons = []
        if state["fraud_risk_score"] >= FRAUD_ESCALATION_THRESHOLD:
            reasons.append(f"elevated fraud risk score ({state['fraud_risk_score']})")
        if state["claim_amount"] >= HIGH_VALUE_CLAIM_THRESHOLD:
            reasons.append(f"high claim value (${state['claim_amount']:,.2f})")
        reason = "Escalated for human review due to " + " and ".join(reasons) + "."
    else:
        decision = "auto_approve"
        reason = "All documents verified, policy eligible, and fraud risk is low."

    return {
        "decision": decision,
        "decision_reason": reason,
        "trace": [f"[Decision Router] decision={decision} -- {reason}"],
    }


def route_after_decision(state: ClaimState) -> str:
    """Conditional-edge function used by the graph to pick the next node."""
    return state["decision"]


# ---------------------------------------------------------------------
# 5. Human Approval Agent (human-in-the-loop)
# ---------------------------------------------------------------------
def human_approval_node(state: ClaimState) -> dict:
    """
    Pauses the graph using LangGraph's `interrupt()`. The Streamlit app
    resumes execution with `Command(resume={"decision": ..., "notes": ...})`
    once a human reviewer submits their verdict.
    """
    payload = interrupt(
        {
            "claim_id": state["claim_id"],
            "claimant_name": state["claimant_name"],
            "claim_summary": state.get("claim_summary", ""),
            "decision_reason": state.get("decision_reason", ""),
            "fraud_risk_score": state.get("fraud_risk_score"),
            "fraud_flags": state.get("fraud_flags", []),
            "claim_amount": state["claim_amount"],
            "message": "Human review required. Please approve or reject this claim.",
        }
    )
    human_decision = payload.get("decision", "rejected")
    human_notes = payload.get("notes", "")

    return {
        "human_decision": human_decision,
        "human_notes": human_notes,
        "final_status": "Approved" if human_decision == "approved" else "Rejected",
        "trace": [f"[Human Approval Agent] human_decision={human_decision}"],
    }


# ---------------------------------------------------------------------
# Terminal nodes for the two non-human-review outcomes
# ---------------------------------------------------------------------
def auto_approve_node(state: ClaimState) -> dict:
    return {
        "final_status": "Approved",
        "trace": ["[Auto Approve] claim approved automatically"],
    }


def auto_reject_node(state: ClaimState) -> dict:
    return {
        "final_status": "Rejected",
        "trace": ["[Auto Reject] claim rejected automatically"],
    }


'''

In [ ]:
# File: state.py

'''

"""
state.py
--------
Shared LangGraph state for the Insurance Claim Processing Agent.

Note on reducers: three nodes (document verification, eligibility check,
fraud detection) run in PARALLEL as fan-out branches from the entry
point and then fan back in to a single `merge` node. Any field that
more than one parallel branch could theoretically write must use an
Annotated reducer so LangGraph can merge concurrent updates instead of
raising an "InvalidUpdateError". We use `operator.add` on the `trace`
list for this purpose; every other field is written by exactly one
node, so plain overwrite is fine.
"""

import operator
from typing import Annotated, List, Optional, TypedDict


class ClaimState(TypedDict, total=False):
    # ---- Input -------------------------------------------------------
    claim_id: str
    policy_number: str
    claimant_name: str
    claim_type: str  # "Auto" | "Health" | "Property" | "Travel"
    claim_amount: float
    policy_coverage_limit: float
    incident_date: str  # ISO date string
    policy_start_date: str
    policy_end_date: str
    prior_claims_last_year: int
    submitted_documents: List[str]

    # ---- Document Verification Agent ---------------------------------
    required_documents: List[str]
    missing_documents: List[str]
    documents_verified: bool
    document_notes: str

    # ---- Eligibility Check Agent --------------------------------------
    eligibility_status: bool
    eligibility_notes: str

    # ---- Fraud Detection Agent -----------------------------------------
    fraud_risk_score: int  # 0-100
    fraud_flags: List[str]
    fraud_notes: str

    # ---- Claim Summary Agent -------------------------------------------
    claim_summary: str

    # ---- Decision / Routing --------------------------------------------
    decision: str  # "auto_approve" | "reject" | "human_review"
    decision_reason: str

    # ---- Human Approval Agent (human-in-the-loop) -----------------------
    human_decision: Optional[str]  # "approved" | "rejected" | None (pending)
    human_notes: Optional[str]

    # ---- Final outcome ---------------------------------------------------
    final_status: str  # "Approved" | "Rejected" | "Awaiting Human Review"

    # ---- Execution trace (parallel-safe via operator.add) ----------------
    trace: Annotated[List[str], operator.add]


'''

In [ ]:
# File: README.md
'''

## Insurance Claim Processing Agent (LangGraph)

An AI-powered insurance claims workflow built with **LangGraph**, wrapped in a **Streamlit** UI. It verifies documents, checks policy eligibility, screens for fraud, summarizes the claim, and routes each case to auto-approval, auto-rejection, or a human reviewer.

### Architecture

```
                          START
             ┌──────────────┼──────────────┐
             ▼              ▼              ▼
   Document Verification  Eligibility   Fraud Detection      <- run in PARALLEL
             └──────────────┼──────────────┘
                            ▼
                          merge
                            ▼
                    Claim Summary Agent
                            ▼
                     Decision Router  (conditional edge)
             ┌──────────────┼──────────────┐
             ▼              ▼              ▼
      auto_approve     auto_reject   Human Approval Agent
             │              │         (interrupt() - HITL)
             ▼              ▼              ▼
            END            END            END
```

#### The 5 required agents/nodes

| # | Agent | File | What it does |
|---|-------|------|---------------|
| 1 | **Document Verification Agent** | `nodes.py::document_verification_node` | Compares submitted documents against the required list for the claim type; flags what's missing. |
| 2 | **Eligibility Check Agent** | `nodes.py::eligibility_check_node` | Confirms the incident date falls within the policy period and the claim type is covered. |
| 3 | **Fraud Detection Agent** | `nodes.py::fraud_detection_node` | Scores fraud risk 0–100 from signals like claim/coverage ratio, how recently the policy started, round-number amounts, and claim history. |
| 4 | **Claim Summary Agent** | `nodes.py::claim_summary_node` | Uses the LLM to write a short underwriter-facing summary combining the three checks above. |
| 5 | **Human Approval Agent** | `nodes.py::human_approval_node` | Pauses the graph with LangGraph's `interrupt()` when a claim is escalated, and resumes once a reviewer submits a decision in the Streamlit UI. |

Two small support nodes (`merge_node`, `decision_node`) handle the parallel fan-in and the rule-based routing logic, respectively - this is what LangGraph's **conditional edges** dispatch on.

#### Parallel execution
`document_verification`, `eligibility_check`, and `fraud_detection` all have an edge directly from `START`, so LangGraph schedules them concurrently. They all edge into a single `merge` node, which only fires once all three branches finish (a standard LangGraph fan-out/fan-in pattern). The `trace` field in state uses an `operator.add` reducer so the three branches can each append to the execution log without conflicting.

#### Human-in-the-loop
`human_approval_node` calls `langgraph.types.interrupt(...)`, which pauses graph execution using the `MemorySaver` checkpointer and returns the payload to the caller. The Streamlit app detects `"__interrupt__"` in the result, renders an Approve/Reject form, and resumes the same thread with `graph.invoke(Command(resume={...}), config)`.

#### Routing rules (`nodes.py::decision_node`)
- Missing documents → **reject**
- Not eligible (expired policy / uncovered claim type) → **reject**
- Fraud score ≥ 90 → **reject** (very high confidence fraud)
- Fraud score ≥ 60, OR claim amount ≥ $15,000 → **human_review**
- Otherwise → **auto_approve**

Thresholds live in `config.py` and are easy to tune.

### LLM usage
Every node calls `llm.py::call_llm(system_prompt, user_prompt)`. The **sidebar** in the Streamlit app lets you pick the provider at runtime - no code edits, no files to touch:

- **Mock (no key needed)** - deterministic rule-based reasoning, so the whole graph and all 5 demo scenarios run end-to-end for free. This is the default.
- **DeepSeek** - paste your own DeepSeek API key into the password-masked field. DeepSeek exposes an OpenAI-compatible endpoint, so this reuses `langchain_openai.ChatOpenAI` pointed at `https://api.deepseek.com` with model `deepseek-chat` (editable in the sidebar).
- **OpenAI** - paste your own OpenAI key, model defaults to `gpt-4o-mini` (editable).

The key you paste is **never written to disk or into any file** - it lives only in that browser session's memory (`st.session_state` → passed straight into the API client for that run). Refreshing the page or closing the tab clears it.

If a call fails (bad key, insufficient quota, network issue), the app shows a clear inline error instead of crashing or silently falling back - so you always know whether you're looking at live-model output or mock output.

You can also set a key via environment variable instead of the UI, for non-Streamlit use (scripts, tests):
```bash
export DEEPSEEK_API_KEY=sk-...
# or
export OPENAI_API_KEY=sk-...
```

### Setup

```bash
pip install -r requirements.txt
streamlit run streamlit_app.py
```

Open the local URL Streamlit prints (usually `http://localhost:8501`).

### Files
- `state.py` - shared `ClaimState` TypedDict (with the parallel-safe `trace` reducer)
- `config.py` - required documents per claim type + fraud/value thresholds
- `llm.py` - LLM wrapper (real OpenAI call or mock fallback)
- `nodes.py` - the 5 agents + decision router + merge/terminal nodes
- `graph.py` - `StateGraph` wiring: parallel fan-out, fan-in, conditional edges, checkpointer
- `streamlit_app.py` - UI: scenario picker, custom claim builder, results dashboard, human review form

### Test scenarios (all verified to produce the expected routing)

| # | Scenario | Key signal | Result |
|---|----------|-----------|--------|
| 1 | Complete claim, valid documents | All docs present, policy active, low fraud score | **Auto Approve** |
| 2 | Missing required documents | Property claim missing repair estimate + ownership proof | **Reject** |
| 3 | Suspicious claim amount | Claim is 94% of coverage limit + policy bought 12 days before incident → fraud score 85 | **Human Review** |
| 4 | Expired policy | Incident date after `policy_end_date` | **Reject** |
| 5 | High-value claim, valid documents | $32,000 claim ≥ $15,000 high-value threshold, otherwise clean | **Human Review** |

All five are selectable from the sidebar dropdown in the Streamlit app ("Preloaded demo scenario" mode), or you can build a fully custom claim ("Custom claim" mode) to test other combinations.

### Notes / possible extensions
- Swap `config.py`'s hard-coded rules for a real policy database lookup.
- Persist the `MemorySaver` checkpointer to a database (e.g. `SqliteSaver`/`PostgresSaver`) so pending human-review claims survive an app restart.
- Add a real document-upload step (PDF/image) and have the Document Verification Agent do OCR-based extraction instead of a checklist match.

'''

### Resume_Screening_RAG_Assistant Folder

In [ ]:
# File: app.py

'''

"""
app.py
------
Streamlit UI for the AI Resume Screening Assistant.

Run with:
    streamlit run app.py

Features:
    - Upload one or more resume PDFs
    - Enter / paste a Job Description
    - Evaluate every resume against the JD via a per-resume RAG pipeline
    - View Match Score, Matching/Missing Skills, Summary, Strengths,
      Weaknesses, and Hiring Recommendation for each candidate
    - Compare any two evaluated candidates side by side
    - Get an auto-ranked "best candidate" recommendation
    - Ask free-form questions about a specific resume (RAG Q&A)
"""

import os
import html as html_lib
import base64
from pathlib import Path

import streamlit as st
import pandas as pd

from rag_engine import build_engine, rank_candidates, PROVIDERS

st.set_page_config(
    page_title="AI Resume Screening Assistant",
    page_icon="",
    layout="wide",
    initial_sidebar_state="expanded",
)

# # --------------------------------------------------------------------------- #
# # Background image (gradient artwork) — swap assets/background.png for any
# # other image to change the look; falls back to a CSS-only gradient if the
# # file isn't found so the app never breaks.
# # --------------------------------------------------------------------------- #
# APP_DIR = Path(__file__).parent
# BG_IMAGE_PATH = APP_DIR / "assets" / "background.png"


@st.cache_data
def _load_base64_image(path_str: str, mtime: float) -> str | None:
    """`mtime` is included purely so the cache key changes whenever the file
    on disk is replaced (even if the filename stays the same) — otherwise
    Streamlit would keep serving stale, previously-cached image bytes.
    (Must NOT be prefixed with an underscore: Streamlit's cache_data excludes
    underscore-prefixed params from the cache key, which would defeat the
    whole point of passing it in.)"""
    p = Path(path_str)
    if not p.exists():
        return None
    with open(p, "rb") as f:
        return base64.b64encode(f.read()).decode()

# --------------------------------------------------------------------------- #
# Design system: colors, icons, and small HTML component builders
# --------------------------------------------------------------------------- #
REC_STYLE = {
    "Strong Hire": {"color": "#16A34A", "bg": "#DCFCE7", "icon": "🌟"},
    "Hire":        {"color": "#2563EB", "bg": "#DBEAFE", "icon": "👍"},
    "Maybe":       {"color": "#D97706", "bg": "#FEF3C7", "icon": "🤔"},
    "No Hire":     {"color": "#DC2626", "bg": "#FEE2E2", "icon": "🚫"},
}
MEDALS = ["🥇", "🥈", "🥉"]


def score_color(score: int) -> str:
    if score >= 80:
        return "#16A34A"   # green
    if score >= 65:
        return "#2563EB"   # blue
    if score >= 45:
        return "#D97706"   # amber
    return "#DC2626"       # red


def esc(text: str) -> str:
    return html_lib.escape(str(text))


def score_ring_html(score: int, size: int = 108) -> str:
    color = score_color(score)
    inner = int(size * 0.78)
    return f"""
    <div style="width:{size}px;height:{size}px;border-radius:50%;
                background:conic-gradient({color} {score * 3.6}deg, #EEF0F6 0deg);
                display:flex;align-items:center;justify-content:center;
                box-shadow:0 2px 10px rgba(30,20,60,0.08);">
      <div style="width:{inner}px;height:{inner}px;border-radius:50%;background:#FFFFFF;
                  display:flex;flex-direction:column;align-items:center;justify-content:center;">
        <span style="font-size:26px;font-weight:800;color:{color};line-height:1;">{score}</span>
        <span style="font-size:10px;font-weight:600;color:#9AA1B4;letter-spacing:0.05em;">/ 100</span>
      </div>
    </div>
    """


def rec_badge_html(recommendation: str) -> str:
    s = REC_STYLE.get(recommendation, {"color": "#6B7280", "bg": "#F3F4F6", "icon": "•"})
    return f"""
    <span style="display:inline-block;padding:6px 14px;border-radius:999px;
                 background:{s['bg']};color:{s['color']};font-weight:700;
                 font-size:13px;letter-spacing:0.01em;">
      {s['icon']}&nbsp; {esc(recommendation)}
    </span>
    """


def chip_row_html(items, kind: str) -> str:
    """kind: 'match' (green) or 'miss' (red/amber)"""
    if not items:
        return "<span style='color:#9AA1B4;font-size:13px;'>None identified</span>"
    palette = (
        {"bg": "#E9FBF0", "fg": "#158A45", "border": "#BCEFD1"}
        if kind == "match"
        else {"bg": "#FDF1F1", "fg": "#C0392B", "border": "#F6D3D0"}
    )
    chips = "".join(
        f"""<span style="display:inline-block;margin:3px 6px 3px 0;padding:5px 12px;
                    border-radius:999px;background:{palette['bg']};color:{palette['fg']};
                    border:1px solid {palette['border']};font-size:12.5px;font-weight:600;">
              {esc(item)}
            </span>"""
        for item in items
    )
    return f"<div>{chips}</div>"


def list_html(items, icon: str) -> str:
    if not items:
        return "<span style='color:#9AA1B4;font-size:13px;'>—</span>"
    rows = "".join(
        f"""<div style="margin:4px 0;font-size:14px;line-height:1.45;">
              <span style="margin-right:6px;">{icon}</span>{esc(item)}
            </div>"""
        for item in items
    )
    return rows


# --------------------------------------------------------------------------- #
# Global CSS
# --------------------------------------------------------------------------- #
st.markdown(
    f"""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

    html, body, [class*="css"]  {{ font-family: 'Inter', sans-serif; }}

    /* Full-app gradient background */
    
    [data-testid="stHeader"] {{
        background: transparent;
    }}

    /* Hero banner — frosted glass over the gradient */
    .hero {{
        background: rgba(255,255,255,0.62);
        backdrop-filter: blur(16px);
        -webkit-backdrop-filter: blur(16px);
        border: 1px solid rgba(255,255,255,0.55);
        padding: 34px 38px;
        border-radius: 18px;
        margin-bottom: 22px;
        box-shadow: 0 10px 30px rgba(108,92,231,0.15);
    }}
    .hero h1 {{
        color: #4B3AA4;
        font-size: 30px;
        font-weight: 800;
        margin: 0 0 6px 0;
    }}
    .hero p {{
        color: #5C5470;
        font-size: 15px;
        margin: 0;
    }}

    /* Candidate card — frosted glass */
    .candidate-card {{
        border: 1px solid rgba(255,255,255,0.6);
        border-radius: 16px;
        padding: 20px 22px;
        margin-bottom: 16px;
        background: rgba(255,255,255,0.72);
        backdrop-filter: blur(14px);
        -webkit-backdrop-filter: blur(14px);
        box-shadow: 0 2px 14px rgba(30,20,60,0.08);
        transition: box-shadow 0.2s ease, transform 0.2s ease;
    }}
    .candidate-card:hover {{
        box-shadow: 0 8px 26px rgba(30,20,60,0.14);
        transform: translateY(-1px);
    }}
    .candidate-name {{
        font-size: 19px;
        font-weight: 800;
        color: #1F2430;
        margin-bottom: 2px;
    }}
    .section-label {{
        font-size: 12.5px;
        font-weight: 700;
        text-transform: uppercase;
        letter-spacing: 0.06em;
        color: #7A7F94;
        margin: 14px 0 6px 0;
    }}
    .divider-soft {{
        border: none;
        border-top: 1px solid rgba(139,143,163,0.25);
        margin: 14px 0;
    }}
    .winner-banner {{
        background: rgba(255,247,230,0.85);
        backdrop-filter: blur(10px);
        border: 1px solid #FBE5B8;
        border-radius: 14px;
        padding: 16px 20px;
        font-size: 15px;
    }}

    /* Sidebar — frosted glass to match */
    section[data-testid="stSidebar"] {{
        background: rgba(255,255,255,0.72);
        backdrop-filter: blur(16px);
        -webkit-backdrop-filter: blur(16px);
        border-right: 1px solid rgba(139,143,163,0.25);
    }}

    /* Buttons */
    div.stButton > button, div.stButton > button:focus {{
        border-radius: 10px;
        font-weight: 700;
        padding: 0.6em 1.2em;
    }}

    /* Tabs */
    button[data-baseweb="tab"] {{
        font-weight: 600;
        font-size: 15px;
    }}
    </style>
    """,
    unsafe_allow_html=True,
)

# --------------------------------------------------------------------------- #
# Session state
# --------------------------------------------------------------------------- #
if "engines" not in st.session_state:
    st.session_state.engines = {}  # name -> ResumeRAGEngine
if "evaluations" not in st.session_state:
    st.session_state.evaluations = {}  # name -> CandidateEvaluation
if "jd_used" not in st.session_state:
    st.session_state.jd_used = ""

# --------------------------------------------------------------------------- #
# Sidebar: configuration
# --------------------------------------------------------------------------- #
with st.sidebar:
    st.markdown("###Configuration")

    provider = st.selectbox(
        "LLM Provider",
        list(PROVIDERS.keys()),
        index=0,  # DeepSeek first (cheap/free-tier friendly)
        help="Pick which chat model powers the evaluation. Embeddings always "
        "run locally for free, regardless of this choice.",
    )
    provider_cfg = PROVIDERS[provider]

    api_key = st.text_input(
        f"{provider} API Key",
        type="password",
        value=os.environ.get(f"{provider.upper()}_API_KEY", ""),
        help="Your key is used only for this session and is never stored.",
    )
    llm_model = st.selectbox("LLM model", provider_cfg["models"], index=0)

    st.markdown(
        """
        <div style="font-size:12.5px;color:#8B8FA3;line-height:1.6;margin-top:6px;">
        - <b>Embeddings:</b> <code>BAAI/bge-small-en-v1.5</code> via FastEmbed
        - runs locally &amp; free, no API key needed.<br>
        - <b>Vector store:</b> FAISS (in-memory, per resume)
        </div>
        """,
        unsafe_allow_html=True,
    )
    st.markdown("<hr class='divider-soft'>", unsafe_allow_html=True)
    if st.button("Clear all data / start over", use_container_width=True):
        st.session_state.engines = {}
        st.session_state.evaluations = {}
        st.session_state.jd_used = ""
        st.rerun()

# --------------------------------------------------------------------------- #
# Hero header
# --------------------------------------------------------------------------- #
st.markdown(
    """
    <div class="hero">
        <h1>AI Resume Screening Assistant</h1>
        <p>LangChain + RAG (FAISS) resume screener — upload resumes, paste a JD,
        get grounded, structured candidate evaluations in seconds.</p>
    </div>
    """,
    unsafe_allow_html=True,
)

# --------------------------------------------------------------------------- #
# Inputs: JD + resumes
# --------------------------------------------------------------------------- #
col_jd, col_files = st.columns([1.2, 1])

with col_jd:
    st.markdown("#### 📋 Job Description")
    jd_text = st.text_area(
        "Job Description",
        height=240,
        placeholder="Paste the full job description here...",
        label_visibility="collapsed",
    )

with col_files:
    st.markdown("#### 📎 Resumes")
    uploaded_files = st.file_uploader(
        "Upload resume PDF(s)",
        type=["pdf"],
        accept_multiple_files=True,
        label_visibility="collapsed",
    )
    if uploaded_files:
        st.markdown(
            f"<div style='font-size:13.5px;color:#4B5066;margin-top:4px;'>"
            f"📄 <b>{len(uploaded_files)}</b> resume(s) selected</div>",
            unsafe_allow_html=True,
        )
        for f in uploaded_files:
            st.markdown(
                f"<div style='font-size:13px;color:#8B8FA3;'>• {esc(f.name)}</div>",
                unsafe_allow_html=True,
            )

st.write("")
evaluate_clicked = st.button(
    "Evaluate resumes against JD", type="primary", use_container_width=True
)

# --------------------------------------------------------------------------- #
# Evaluation run
# --------------------------------------------------------------------------- #
if evaluate_clicked:
    if not api_key:
        st.error(f"Please enter your {provider} API key in the sidebar.")
    elif not jd_text.strip():
        st.error("Please paste a job description.")
    elif not uploaded_files:
        st.error("Please upload at least one resume PDF.")
    else:
        st.session_state.jd_used = jd_text
        progress = st.progress(0.0, text="Starting evaluation...")
        n = len(uploaded_files)
        for i, uf in enumerate(uploaded_files):
            name = os.path.splitext(uf.name)[0]
            progress.progress(
                i / n, text=f"🔎 Indexing & evaluating {uf.name} ({i+1}/{n})..."
            )
            try:
                engine = build_engine(
                    uf, api_key, llm_model, base_url=provider_cfg["base_url"]
                )
                evaluation = engine.evaluate(jd_text)
                st.session_state.engines[name] = engine
                st.session_state.evaluations[name] = evaluation
            except Exception as e:
                st.error(f"Failed to process {uf.name}: {e}")
        progress.progress(1.0, text="Done.")
        st.success(f"✅ Evaluated {len(st.session_state.evaluations)} resume(s).")

# --------------------------------------------------------------------------- #
# Results
# --------------------------------------------------------------------------- #
evals = st.session_state.evaluations

if evals:
    tab_individual, tab_compare, tab_rank = st.tabs(
        ["📄 Individual Evaluations", "⚖️ Compare Two", "🏆 Rank & Recommend"]
    )

    # ----------------------------- Individual ----------------------------- #
    with tab_individual:
        for name, ev in evals.items():
            with st.container():
                st.markdown('<div class="candidate-card">', unsafe_allow_html=True)

                head_l, head_r = st.columns([3, 1])
                with head_l:
                    st.markdown(
                        f'<div class="candidate-name">👤 {esc(ev.candidate_name)}</div>',
                        unsafe_allow_html=True,
                    )
                    st.markdown(rec_badge_html(ev.recommendation), unsafe_allow_html=True)
                    st.markdown(
                        f"<div style='margin-top:10px;font-size:14px;color:#4B5066;"
                        f"line-height:1.5;'>📝 {esc(ev.summary)}</div>",
                        unsafe_allow_html=True,
                    )
                    st.markdown(
                        f"<div style='margin-top:10px;font-size:13px;color:#8B8FA3;"
                        f"font-style:italic;'>💡 {esc(ev.justification)}</div>",
                        unsafe_allow_html=True,
                    )
                with head_r:
                    st.markdown(
                        f"<div style='display:flex;justify-content:center;'>"
                        f"{score_ring_html(ev.match_score)}</div>",
                        unsafe_allow_html=True,
                    )

                st.markdown("<hr class='divider-soft'>", unsafe_allow_html=True)

                sc1, sc2 = st.columns(2)
                with sc1:
                    st.markdown('<div class="section-label">✅ Matching Skills</div>', unsafe_allow_html=True)
                    st.markdown(chip_row_html(ev.matching_skills, "match"), unsafe_allow_html=True)
                    st.markdown('<div class="section-label">💪 Strengths</div>', unsafe_allow_html=True)
                    st.markdown(list_html(ev.strengths, "✔️"), unsafe_allow_html=True)
                with sc2:
                    st.markdown('<div class="section-label">🚫 Missing Skills</div>', unsafe_allow_html=True)
                    st.markdown(chip_row_html(ev.missing_skills, "miss"), unsafe_allow_html=True)
                    st.markdown('<div class="section-label">⚠️ Weaknesses</div>', unsafe_allow_html=True)
                    st.markdown(list_html(ev.weaknesses, "•"), unsafe_allow_html=True)

                st.markdown("</div>", unsafe_allow_html=True)

    # ----------------------------- Compare -------------------------------- #
    with tab_compare:
        names = list(evals.keys())
        if len(names) < 2:
            st.info("Upload and evaluate at least two resumes to compare.")
        else:
            c1, c2 = st.columns(2)
            with c1:
                name_a = st.selectbox("👤 Candidate A", names, index=0, key="cmp_a")
            with c2:
                name_b = st.selectbox(
                    "👤 Candidate B", names, index=min(1, len(names) - 1), key="cmp_b"
                )

            if name_a and name_b:
                ea, eb = evals[name_a], evals[name_b]
                winner = ea if ea.match_score >= eb.match_score else eb

                cc1, cc2 = st.columns(2)
                for col, ev in ((cc1, ea), (cc2, eb)):
                    crown = "👑 " if ev.candidate_name == winner.candidate_name else ""
                    with col:
                        st.markdown('<div class="candidate-card">', unsafe_allow_html=True)
                        st.markdown(
                            f'<div class="candidate-name">{crown}{esc(ev.candidate_name)}</div>',
                            unsafe_allow_html=True,
                        )
                        st.markdown(
                            f"<div style='display:flex;justify-content:center;margin:10px 0;'>"
                            f"{score_ring_html(ev.match_score, size=92)}</div>",
                            unsafe_allow_html=True,
                        )
                        st.markdown(
                            f"<div style='display:flex;justify-content:center;'>"
                            f"{rec_badge_html(ev.recommendation)}</div>",
                            unsafe_allow_html=True,
                        )
                        st.markdown("<hr class='divider-soft'>", unsafe_allow_html=True)
                        st.markdown('<div class="section-label">✅ Matching Skills</div>', unsafe_allow_html=True)
                        st.markdown(chip_row_html(ev.matching_skills, "match"), unsafe_allow_html=True)
                        st.markdown('<div class="section-label">🚫 Missing Skills</div>', unsafe_allow_html=True)
                        st.markdown(chip_row_html(ev.missing_skills, "miss"), unsafe_allow_html=True)
                        st.markdown("</div>", unsafe_allow_html=True)

                st.markdown(
                    f"""<div class="winner-banner">
                        🏆 For this JD, <b>{esc(winner.candidate_name)}</b> has the
                        stronger match (<b>{winner.match_score}/100</b>, {esc(winner.recommendation)}).
                        </div>""",
                    unsafe_allow_html=True,
                )

    # ------------------------------- Rank ---------------------------------- #
    with tab_rank:
        ranked = rank_candidates(list(evals.values()))

        if ranked:
            best = ranked[0]
            st.markdown(
                f"""<div class="winner-banner" style="margin-bottom:18px;">
                    🏆 <b>Best candidate: {esc(best.candidate_name)}</b>
                    &nbsp;·&nbsp; Score {best.match_score}/100 &nbsp;·&nbsp; {esc(best.recommendation)}
                    <div style="margin-top:6px;font-size:13.5px;color:#4B5066;">{esc(best.justification)}</div>
                    </div>""",
                unsafe_allow_html=True,
            )

        for i, ev in enumerate(ranked):
            medal = MEDALS[i] if i < 3 else f"#{i+1}"
            bar_color = score_color(ev.match_score)
            with st.container():
                st.markdown('<div class="candidate-card">', unsafe_allow_html=True)
                r1, r2, r3 = st.columns([0.4, 2.6, 1])
                with r1:
                    st.markdown(
                        f"<div style='font-size:26px;text-align:center;'>{medal}</div>",
                        unsafe_allow_html=True,
                    )
                with r2:
                    st.markdown(
                        f"<div class='candidate-name' style='margin-bottom:6px;'>{esc(ev.candidate_name)}</div>",
                        unsafe_allow_html=True,
                    )
                    st.markdown(
                        f"""<div style="background:#F0EEFA;border-radius:8px;height:10px;width:100%;">
                            <div style="background:{bar_color};width:{ev.match_score}%;height:10px;
                                        border-radius:8px;"></div>
                            </div>""",
                        unsafe_allow_html=True,
                    )
                    st.markdown(
                        f"<div style='margin-top:8px;'>{rec_badge_html(ev.recommendation)}</div>",
                        unsafe_allow_html=True,
                    )
                with r3:
                    st.markdown(
                        f"<div style='text-align:center;font-size:24px;font-weight:800;color:{bar_color};'>"
                        f"{ev.match_score}<span style='font-size:13px;color:#9AA1B4;'>/100</span></div>",
                        unsafe_allow_html=True,
                    )
                st.markdown("</div>", unsafe_allow_html=True)

else:
    st.markdown(
        """
        <div class="candidate-card" style="text-align:center;padding:40px;">
            <div style="font-size:40px;">🚀</div>
            <div style="font-size:16px;font-weight:700;margin-top:8px;color:#1F2430;">
                Ready when you are
            </div>
            <div style="font-size:14px;color:#8B8FA3;margin-top:4px;">
                Paste a Job Description, upload one or more resume PDFs, then click
                <b>Evaluate resumes against JD</b> to get started.
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


'''

In [ ]:
# File: config.toml

'''

[theme]
base = "light"
primaryColor = "#6C5CE7"
backgroundColor = "#FFFFFF"
secondaryBackgroundColor = "#F5F3FF"
textColor = "#1F2430"
font = "sans serif"

[browser]
gatherUsageStats = false


'''

In [ ]:
# File: generate_sample_resumes.py

'''

"""
Generates three sample resume PDFs (Resume_A, Resume_B, Resume_C) used to
demo the AI Resume Screening Assistant against sample_data/jd_data_scientist.txt.

Run once:  python generate_sample_resumes.py
"""

import os
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

OUT_DIR = os.path.dirname(os.path.abspath(__file__))

styles = getSampleStyleSheet()
h1 = ParagraphStyle("h1", parent=styles["Heading1"], spaceAfter=4)
h2 = ParagraphStyle("h2", parent=styles["Heading2"], spaceBefore=10, spaceAfter=4)
body = ParagraphStyle("body", parent=styles["Normal"], spaceAfter=4, leading=14)


def build_resume(filename: str, sections: list[tuple[str, str]]):
    path = os.path.join(OUT_DIR, filename)
    doc = SimpleDocTemplate(
        path, pagesize=letter,
        leftMargin=0.8 * inch, rightMargin=0.8 * inch,
        topMargin=0.7 * inch, bottomMargin=0.7 * inch,
    )
    story = []
    for i, (heading, text) in enumerate(sections):
        style = h1 if i == 0 else h2
        story.append(Paragraph(heading, style))
        for line in text.strip().split("\n"):
            story.append(Paragraph(line.strip(), body))
        story.append(Spacer(1, 6))
    doc.build(story)
    print(f"Wrote {path}")


# --------------------------------------------------------------------------- #
# Resume A: strong Data Scientist match
# --------------------------------------------------------------------------- #
resume_a = [
    ("Priya Sharma", "Email: priya.sharma@example.com | Phone: (555) 111-2222"),
    ("Summary", """
    Data Scientist with 5 years of experience building and deploying machine
    learning models for e-commerce and fintech companies. Skilled in Python,
    deep learning, and cloud-based ML deployment.
    """),
    ("Skills", """
    Python, pandas, NumPy, scikit-learn, TensorFlow, PyTorch, SQL,
    A/B testing and experimental design, AWS (SageMaker, S3, EC2),
    Matplotlib, Seaborn, Tableau, MLflow, Airflow, Docker, Spark
    """),
    ("Experience", """
    Senior Data Scientist, FinEdge Analytics (2022-Present)
    - Built a churn prediction model using scikit-learn and TensorFlow,
      improving retention by 12%.
    - Deployed models to production on AWS SageMaker with MLflow tracking
      and Airflow-orchestrated retraining pipelines.
    - Designed and analyzed A/B tests for pricing experiments.

    Data Scientist, ShopWave Inc. (2020-2022)
    - Built recommendation systems using PyTorch, increasing conversion by 8%.
    - Used Spark for large-scale feature engineering on 500M+ row datasets.
    - Presented insights to executive stakeholders via Tableau dashboards.
    """),
    ("Education", """
    M.S. in Computer Science, University of Washington (2020)
    B.S. in Statistics, University of Michigan (2018)
    """),
]

# --------------------------------------------------------------------------- #
# Resume B: moderate match - strong SWE, lighter on ML/stats
# --------------------------------------------------------------------------- #
resume_b = [
    ("James Carter", "Email: james.carter@example.com | Phone: (555) 333-4444"),
    ("Summary", """
    Backend Software Engineer with 4 years of experience building scalable
    web services. Recently transitioning toward data-focused roles, with
    some exposure to Python-based data analysis.
    """),
    ("Skills", """
    Python, Java, SQL, PostgreSQL, REST APIs, Docker, Kubernetes,
    pandas (basic), Git, CI/CD, AWS (EC2, Lambda)
    """),
    ("Experience", """
    Software Engineer, CloudBridge Systems (2021-Present)
    - Built and maintained REST APIs serving 2M+ daily requests.
    - Wrote internal Python scripts using pandas for weekly reporting.
    - Deployed services on AWS using Docker and Kubernetes.

    Junior Software Engineer, DataPort LLC (2020-2021)
    - Maintained SQL-based ETL pipelines feeding internal dashboards.
    - Collaborated with the analytics team on ad-hoc data pulls.
    """),
    ("Education", """
    B.S. in Computer Science, Ohio State University (2020)
    """),
]

# --------------------------------------------------------------------------- #
# Resume C: weak match - entry-level analyst, many missing skills
# --------------------------------------------------------------------------- #
resume_c = [
    ("Maria Lopez", "Email: maria.lopez@example.com | Phone: (555) 777-8888"),
    ("Summary", """
    Recent graduate with a background in business analytics and Excel-based
    reporting. Eager to grow into a data-focused role.
    """),
    ("Skills", """
    Excel, PowerPoint, basic SQL, Google Sheets, PowerBI (beginner),
    strong communication and presentation skills
    """),
    ("Experience", """
    Business Analyst Intern, RetailNow Corp. (Summer 2025)
    - Built weekly sales reports in Excel and PowerBI for the merchandising team.
    - Wrote basic SQL queries to pull data from the company's data warehouse.
    - Assisted with slide decks summarizing quarterly performance for leadership.
    """),
    ("Education", """
    B.A. in Business Administration, Concentration in Analytics,
    Arizona State University (2025)
    """),
]

if __name__ == "__main__":
    build_resume("Resume_A.pdf", resume_a)
    build_resume("Resume_B.pdf", resume_b)
    build_resume("Resume_C.pdf", resume_c)


'''

In [ ]:
# File: rag_engine.py

'''

"""
rag_engine.py
--------------
Core RAG (Retrieval-Augmented Generation) engine for the AI Resume Screening Assistant.

Pipeline for each resume:
    PDF  --(PyPDFLoader)-->  raw text
         --(RecursiveCharacterTextSplitter)-->  chunks
         --(FastEmbedEmbeddings, local & free)-->  vectors
         --(FAISS)-->  vector store
         --(retriever)-->  JD-relevant chunks
         --(ChatPromptTemplate + structured LLM)-->  CandidateEvaluation (Pydantic)

Everything the LLM says about a candidate is grounded ONLY in chunks retrieved
from that candidate's resume + the job description supplied by the recruiter.

LLM provider is pluggable: OpenAI or any OpenAI-compatible endpoint (e.g.
DeepSeek). Embeddings always run locally via FastEmbed (ONNX, CPU, free, no
API key) so indexing never depends on - or costs money against - the chat
provider's quota.
"""

from __future__ import annotations

import os
import tempfile
from typing import List, Literal, Optional

from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document


# --------------------------------------------------------------------------- #
# 0. Supported LLM providers (all speak the OpenAI-compatible chat API)
# --------------------------------------------------------------------------- #
PROVIDERS = {
    "DeepSeek": {
        "base_url": "https://api.deepseek.com",
        "default_model": "deepseek-chat",
        "models": ["deepseek-chat", "deepseek-reasoner"],
    },
    "OpenAI": {
        "base_url": None,  # use the SDK default
        "default_model": "gpt-4o-mini",
        "models": ["gpt-4o-mini", "gpt-4o", "gpt-4.1-mini"],
    },
}

# Cached embedder - loading the ONNX model is the slow part, so every
# ResumeRAGEngine instance in the session reuses the same one.
_EMBEDDER: Optional[FastEmbedEmbeddings] = None


def get_embedder() -> FastEmbedEmbeddings:
    global _EMBEDDER
    if _EMBEDDER is None:
        _EMBEDDER = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")
    return _EMBEDDER


# --------------------------------------------------------------------------- #
# 1. Structured output schema (the "Output Parser" requirement)
# --------------------------------------------------------------------------- #
class CandidateEvaluation(BaseModel):
    """Structured evaluation of a single candidate against a Job Description."""

    candidate_name: str = Field(
        description="Candidate's name as found in the resume. Use the filename "
        "(without extension) if no name can be identified."
    )
    match_score: int = Field(
        description="Overall match score between 0 and 100, where 100 is a "
        "perfect match to the job description.",
        ge=0,
        le=100,
    )
    matching_skills: List[str] = Field(
        description="Skills/technologies/qualifications required by the JD that "
        "ARE evidenced in the resume."
    )
    missing_skills: List[str] = Field(
        description="Skills/technologies/qualifications required by the JD that "
        "are NOT evidenced in the resume."
    )
    summary: str = Field(
        description="A concise 2-4 sentence summary of the candidate's background "
        "as it relates to the JD."
    )
    strengths: List[str] = Field(
        description="Concrete strengths of this candidate for this specific role."
    )
    weaknesses: List[str] = Field(
        description="Concrete weaknesses, gaps, or risk areas for this specific role."
    )
    recommendation: Literal["Strong Hire", "Hire", "Maybe", "No Hire"] = Field(
        description="Overall hiring recommendation."
    )
    justification: str = Field(
        description="1-3 sentence justification for the recommendation, referencing "
        "specific evidence from the resume."
    )


# --------------------------------------------------------------------------- #
# 2. Prompt template
# --------------------------------------------------------------------------- #
EVALUATION_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert technical recruiter and resume screening assistant. "
            "You evaluate ONLY based on the resume excerpts provided to you as context "
            "- never invent experience, skills, or credentials that are not present in "
            "the context. If information is not present in the context, treat it as "
            "absent/unknown rather than assuming it exists.\n\n"
            "You will be given:\n"
            "1. A Job Description (JD)\n"
            "2. Retrieved excerpts from ONE candidate's resume (via RAG)\n\n"
            "Produce a rigorous, evidence-based evaluation of this candidate against "
            "the JD, following the required output schema exactly.",
        ),
        (
            "human",
            "JOB DESCRIPTION:\n{jd}\n\n"
            "RETRIEVED RESUME EXCERPTS (candidate: {source_name}):\n{context}\n\n"
            "Evaluate this candidate against the job description above.",
        ),
    ]
)

# Sub-queries used to pull a well-rounded set of chunks out of the resume,
# rather than relying on a single similarity search against the raw JD text.
RETRIEVAL_QUERIES_TEMPLATE = [
    "{jd}",
    "technical skills, tools, technologies, and programming languages",
    "work experience, job titles, responsibilities, and achievements",
    "education, degrees, certifications, and qualifications",
]


class ResumeRAGEngine:
    """
    One instance manages the RAG pipeline for a single uploaded resume:
    load -> split -> embed -> index -> retrieve -> evaluate.
    """

    def __init__(
        self,
        pdf_path: str,
        source_name: str,
        api_key: str,
        llm_model: str = "deepseek-chat",
        base_url: Optional[str] = "https://api.deepseek.com",
        chunk_size: int = 800,
        chunk_overlap: int = 150,
        retriever_k: int = 4,
    ):
        self.pdf_path = pdf_path
        self.source_name = source_name
        self.retriever_k = retriever_k

        # Embeddings are always local/free (FastEmbed, ONNX, CPU) - this
        # keeps indexing working even for providers (like DeepSeek) that
        # don't expose an embeddings endpoint at all.
        self.embeddings = get_embedder()

        self.llm = ChatOpenAI(
            model=llm_model,
            temperature=0,
            api_key=api_key,
            base_url=base_url,  # None => official OpenAI endpoint
        )
        # method="function_calling" is used instead of the default
        # strict-JSON-schema mode, since that stricter mode is an
        # OpenAI-only feature that non-OpenAI endpoints (DeepSeek, etc.)
        # don't support - function calling is broadly compatible.
        self.structured_llm = self.llm.with_structured_output(
            CandidateEvaluation, method="function_calling"
        )

        self.chunks: List[Document] = self._load_and_split(chunk_size, chunk_overlap)
        self.vectorstore = FAISS.from_documents(self.chunks, self.embeddings)
        self.retriever = self.vectorstore.as_retriever(
            search_type="similarity", search_kwargs={"k": self.retriever_k}
        )

    # ------------------------------------------------------------------ #
    # Loading + splitting
    # ------------------------------------------------------------------ #
    def _load_and_split(self, chunk_size: int, chunk_overlap: int) -> List[Document]:
        loader = PyPDFLoader(self.pdf_path)
        pages = loader.load()

        for p in pages:
            p.metadata["source"] = self.source_name

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
        return splitter.split_documents(pages)

    # ------------------------------------------------------------------ #
    # Retrieval
    # ------------------------------------------------------------------ #
    def retrieve_context(self, jd: str) -> str:
        """Multi-query retrieval: pull chunks relevant to the JD as a whole,
        plus chunks specifically about skills / experience / education, then
        dedupe. This gives the LLM a well-rounded, grounded view of the resume
        rather than only the single most JD-similar paragraph."""
        seen = set()
        collected: List[Document] = []

        for template in RETRIEVAL_QUERIES_TEMPLATE:
            query = template.format(jd=jd)
            for doc in self.retriever.invoke(query):
                key = doc.page_content[:120]
                if key not in seen:
                    seen.add(key)
                    collected.append(doc)

        if not collected:
            return "(No relevant content retrieved from resume.)"

        return "\n\n---\n\n".join(
            f"[chunk {i+1}]\n{doc.page_content}" for i, doc in enumerate(collected)
        )

    # ------------------------------------------------------------------ #
    # Evaluation (RAG generation step)
    # ------------------------------------------------------------------ #
    def evaluate(self, jd: str) -> CandidateEvaluation:
        context = self.retrieve_context(jd)
        chain = EVALUATION_PROMPT | self.structured_llm
        result: CandidateEvaluation = chain.invoke(
            {"jd": jd, "context": context, "source_name": self.source_name}
        )
        return result

    def answer_question(self, question: str) -> str:
        """Free-form RAG Q&A over this single resume (answers only from
        retrieved resume content)."""
        docs = self.retriever.invoke(question)
        context = "\n\n---\n\n".join(d.page_content for d in docs) or "(no content)"
        qa_prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "Answer the question using ONLY the resume excerpts provided. "
                    "If the answer is not in the excerpts, say you don't have "
                    "enough information in the resume to answer.",
                ),
                ("human", "Resume excerpts:\n{context}\n\nQuestion: {question}"),
            ]
        )
        chain = qa_prompt | self.llm
        return chain.invoke({"context": context, "question": question}).content


# --------------------------------------------------------------------------- #
# 3. Convenience helpers used by the Streamlit app
# --------------------------------------------------------------------------- #
def save_uploaded_pdf(uploaded_file) -> str:
    """Persist a Streamlit UploadedFile to a temp path and return that path."""
    suffix = ".pdf"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(uploaded_file.getbuffer())
        return tmp.name


def build_engine(
    uploaded_file,
    api_key: str,
    llm_model: str = "deepseek-chat",
    base_url: Optional[str] = "https://api.deepseek.com",
) -> ResumeRAGEngine:
    path = save_uploaded_pdf(uploaded_file)
    name = os.path.splitext(uploaded_file.name)[0]
    return ResumeRAGEngine(
        pdf_path=path,
        source_name=name,
        api_key=api_key,
        llm_model=llm_model,
        base_url=base_url,
    )


def rank_candidates(
    evaluations: List[CandidateEvaluation],
) -> List[CandidateEvaluation]:
    """Sort candidates best-to-worst by match_score (ties broken by
    recommendation strength)."""
    rec_order = {"Strong Hire": 3, "Hire": 2, "Maybe": 1, "No Hire": 0}
    return sorted(
        evaluations,
        key=lambda e: (e.match_score, rec_order.get(e.recommendation, 0)),
        reverse=True,
    )


'''

In [ ]:
#  File: jd_data_scientist.txt

'''

Job Title: Data Scientist

We are looking for a Data Scientist to join our analytics team. The ideal
candidate will have strong experience building machine learning models,
working with large datasets, and communicating insights to stakeholders.

Required Skills:
- Strong proficiency in Python (pandas, NumPy, scikit-learn)
- Experience with SQL and relational databases
- Hands-on experience building and deploying machine learning models
- Experience with deep learning frameworks (TensorFlow or PyTorch)
- Solid understanding of statistics and experimental design (A/B testing)
- Experience with cloud platforms (AWS, GCP, or Azure)
- Strong data visualization skills (Matplotlib, Seaborn, Tableau, or PowerBI)
- Bachelor's or Master's degree in Computer Science, Statistics, or related field

Nice to Have:
- Experience with MLOps tools (MLflow, Airflow, Docker, Kubernetes)
- Experience with NLP or computer vision
- Experience with big data tools (Spark, Hadoop)
- Prior experience in a fast-paced startup environment

Responsibilities:
- Design and build predictive models to support business decisions
- Partner with product and engineering teams to deploy models to production
- Analyze large datasets to extract actionable insights
- Present findings to non-technical stakeholders
- Own the full ML lifecycle from data exploration to monitoring in production


'''

In [ ]:
# File: README.md

'''

## AI Resume Screening Assistant (LangChain + RAG)

A Streamlit app that lets a recruiter upload resumes (PDF), paste a Job
Description, and get a grounded, structured evaluation of each candidate:
**Match Score, Matching/Missing Skills, Summary, Strengths, Weaknesses, and
a Hiring Recommendation** - generated only from what's actually retrieved
out of each resume via RAG.

### Architecture

```
Resume PDF ──► PyPDFLoader ──► RecursiveCharacterTextSplitter ──► chunks
                                                                     │
                                                FastEmbedEmbeddings (local, free)
                                                                     │
                                                                     ▼
                                                          FAISS vector store
                                                             (per resume)
                                                                     │
Job Description ──► multi-query retrieval (JD + skills/exp/education) ──► retriever
                                                                     │
                                                                     ▼
                                ChatPromptTemplate + ChatOpenAI (DeepSeek or OpenAI)
                                .with_structured_output(CandidateEvaluation)
                                                                     │
                                                                     ▼
                                        Pydantic-validated JSON evaluation
                                                                     │
                                                                     ▼
                                              Streamlit UI (cards, compare, rank)
```

**LLM provider is pluggable** - pick **DeepSeek** (cheap/low-cost, OpenAI-compatible
API) or **OpenAI** from the sidebar. **Embeddings always run locally for free**
via FastEmbed (ONNX, CPU, no API key, no per-resume cost) regardless of which
LLM provider you choose - this matters because DeepSeek doesn't expose an
embeddings endpoint at all.

| Requirement | Implementation |
|---|---|
| LangChain | `langchain`, `langchain-core`, `langchain-community`, `langchain-openai` |
| LLM | `ChatOpenAI` pointed at either `https://api.deepseek.com` (`deepseek-chat`) or the default OpenAI endpoint (`gpt-4o-mini`), switchable in the sidebar |
| PDF Document Loader | `PyPDFLoader` |
| Text Splitter | `RecursiveCharacterTextSplitter` |
| Embedding Model | `FastEmbedEmbeddings` (`BAAI/bge-small-en-v1.5`) - local, free, no API key |
| Vector Database | `FAISS` (one in-memory index per resume) |
| Retriever | `vectorstore.as_retriever()`, multi-query (JD + skills/exp/education sub-queries) |
| Prompt Template | `ChatPromptTemplate` (system + human messages) |
| Output Parser | Pydantic `CandidateEvaluation` model via `.with_structured_output(..., method="function_calling")` |
| Streamlit app | `app.py` |

> `method="function_calling"` is used instead of the newer strict JSON-schema
> mode because that stricter mode is OpenAI-only - function calling is
> supported by both OpenAI and DeepSeek, so evaluations work either way.

Each resume gets **its own vector store**, so evaluations are strictly
grounded in that candidate's own content - nothing is cross-contaminated
between candidates, and the model is explicitly instructed not to invent
skills or experience that aren't in the retrieved context.

### Project Structure

```
resume_screener/
├── app.py                          # Streamlit UI
├── rag_engine.py                   # RAG pipeline, schema, prompt
├── requirements.txt
├── README.md
├── .streamlit/
│   └── config.toml                 # Custom violet theme (consistent look regardless of system dark/light mode)
├── assets/
│   ├── background.png              # Active gradient background image
│   └── background_alt.png          # Alternate gradient background (see "Design" below)
└── sample_data/
    ├── jd_data_scientist.txt       # Sample JD for testing
    ├── generate_sample_resumes.py  # Regenerates the sample PDFs below
    ├── Resume_A.pdf                # Strong match (Data Scientist)
    ├── Resume_B.pdf                # Moderate match (Backend SWE)
    └── Resume_C.pdf                # Weak match (Business Analyst intern)
```

### Setup

```bash
python3 -m venv venv
source venv/bin/activate        # Windows: venv\Scripts\activate
pip install -r requirements.txt

streamlit run app.py
```

Open the URL Streamlit prints (usually `http://localhost:8501`), then in the
sidebar pick a **Provider** (DeepSeek or OpenAI) and paste the matching API
key. Get a DeepSeek key at https://platform.deepseek.com/api_keys - no
OpenAI account is required.

(Optional: instead of pasting the key each time, set `DEEPSEEK_API_KEY` or
`OPENAI_API_KEY` as an environment variable before launching, and the sidebar
will pre-fill it.)

The first run will download the local embedding model (~130MB, one-time,
requires normal internet access) - after that it's cached and runs offline.

### Using the Sample Data

1. In the sidebar, pick a provider and enter your API key.
2. Copy the contents of `sample_data/jd_data_scientist.txt` into the **Job
   Description** box.
3. Upload `Resume_A.pdf`, `Resume_B.pdf`, and `Resume_C.pdf` from
   `sample_data/`.
4. Click **Evaluate resumes against JD**.

(To regenerate the sample PDFs: `python sample_data/generate_sample_resumes.py`.)

### Example Test Cases

All five example scenarios from the brief are covered by the app's tabs:

1. **Evaluate Resume A for a Data Scientist role**
   → Load the JD + `Resume_A.pdf` only, click Evaluate, open the
   **Individual Evaluations** tab. Expect a high match score - Resume A
   covers Python/scikit-learn/TensorFlow/PyTorch, AWS, MLflow/Airflow, A/B
   testing, and an M.S. in Computer Science.

2. **Compare Resume A and Resume B for the same JD**
   → Upload both, evaluate, go to the **Compare Two** tab, pick Resume A vs
   Resume B. Resume A (data scientist) should outscore Resume B (backend
   engineer with only light pandas/AWS exposure and no ML/stats background).

3. **Identify missing skills in Resume C**
   → Evaluate `Resume_C.pdf`, open its card in **Individual Evaluations**
   and read the **Missing Skills** field. Expect gaps like Python, machine
   learning frameworks (TensorFlow/PyTorch), SQL depth, A/B testing/statistics,
   and cloud platforms - Resume C is an Excel/PowerBI business-analyst intern.

4. **Recommend the best candidate among multiple resumes**
   → Upload all three, evaluate, open the **Rank & Recommend** tab. Candidates
   are sorted by match score (ties broken by recommendation strength), with
   the top candidate highlighted along with the model's justification.

5. **Generate a hiring recommendation with justification**
   → Every card in **Individual Evaluations** shows a `recommendation`
   (`Strong Hire` / `Hire` / `Maybe` / `No Hire`) plus a `justification`
   string citing specific resume evidence.

### Design

The UI uses a full-page gradient background image (`assets/background.png`)
with a frosted-glass ("glassmorphism") look layered on top for readability:
- The hero header, candidate cards, and sidebar all use a semi-transparent
  white background + backdrop blur, so text stays crisp over the busy image
- A light white gradient overlay is baked into the background CSS so the
  image never fights with text contrast
- Circular score "rings" color-coded by band (green ≥80, blue 65-79, amber 45-64, red <45)
- Pill-shaped skill chips (green = matching, red = missing)
- Colored recommendation badges (🌟 Strong Hire, 👍 Hire, 🤔 Maybe, 🚫 No Hire)
- Card-based layout with hover lift, used consistently across all four tabs
- Medal icons (🥇🥈🥉) + horizontal score bars in the ranking leaderboard

**To change the background image**: replace `assets/background.png` with any
image of the same name (a second option, `assets/background_alt.png`, is
included - just rename it to `background.png` to switch). If the file is
ever missing, `app.py` automatically falls back to a CSS-only gradient so the
app never breaks.

### Notes / Design Decisions

- **Multi-query retrieval**: instead of a single similarity search against
  the raw JD, the retriever runs against the JD text *and* three targeted
  sub-queries (skills, experience, education) so the LLM sees a well-rounded
  slice of the resume rather than only the single most JD-similar paragraph.
- **Grounding**: the system prompt explicitly forbids inventing skills or
  experience not present in the retrieved context, and untouched information
  is treated as absent rather than assumed.
- **Structured output**: `ChatOpenAI.with_structured_output(CandidateEvaluation)`
  guarantees a schema-valid Pydantic object every time (score bounds,
  enum-constrained recommendation, typed lists) - no manual JSON parsing.
- **Isolation**: each resume gets its own FAISS index, so retrieval for
  Candidate A can never leak Candidate B's content into the same evaluation.
- **Swap-able models**: `gpt-4o-mini` is the default per the brief, but the
  sidebar lets you switch to `gpt-4o` / `gpt-4.1-mini`; you could similarly
  swap `OpenAIEmbeddings`/`FAISS` for `HuggingFaceEmbeddings`/`Chroma` inside
  `rag_engine.py` without touching `app.py`.

'''

### Requirements.txt

In [ ]:
"""
streamlit>=1.36
langchain>=0.2.16
langchain-core>=0.2.38
langgraph>=1.2.10
langchain-community>=0.2.16
langchain-openai>=1.4.1
langchain-groq>=1.1.3
langchain-google-genai>=4.3.2
langchain-deepseek>=1.1.0
langchain-text-splitters>=0.2.4
faiss-cpu>=1.8.0
pypdf>=4.3.1
pydantic>=2.7
pandas>=2.3.3
openai>=1.40.0
tiktoken>=0.7.0
numpy>=2.4.3
fastembed>=0.3.4
reportlab>=5.0.1

"""